<!-- cabecera-entorno -->
## Antes de empezar

**Clase 3 · Limpieza de datos** — Bloque 2 · Demo. Este cuaderno se recorre **por su cuenta**: explica cada concepto antes de usarlo y define cada término la primera vez que aparece. El profesor circula por el salón resolviendo dudas. No hay que esperar a que alguien lo dicte.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El cuaderno se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/educacion_estadisticas.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 3 · Demo — Limpieza de datos

> **Dónde vamos.** Segunda de las cuatro clases de **EDA**, análisis exploratorio de datos (2, 3,
> 4 y 5). Sus cuatro verbos son entender, limpiar y organizar, describir y relacionar: la clase 2
> fue entender y **hoy toca limpiar y organizar**, porque lo que no está limpio no se puede
> describir sin mentir. El marco
> completo está en el cuaderno de la clase 2, sección *Qué estamos haciendo: EDA*.

**Dataset:** `../datos/educacion_estadisticas.csv`

## Cómo se usa este cuaderno

Usted avanza solo, leyendo. Cada bloque de código viene precedido de la explicación del concepto
que usa, y cada término nuevo se define la primera vez que aparece.

**El recorrido:**

| Sección | De qué va | Qué se lleva |
|---------|-----------|--------------|
| 0 y 1 | Qué es limpiar datos y qué tiene de sucio este archivo | El diagnóstico antes de tocar nada |
| 2 | La deuda de la clase 2: `.str.contains()` y `.query()` | Dos formas de filtrar que faltaban |
| 3 | Paso 1 · Inspeccionar | El ritual de cinco comandos |
| 4 | Paso 2 · Nulos | El marco de decisión, y la justificación escrita |
| 5 | Paso 3 · Tipos | `to_numeric`, `astype`, y la regla de orden |
| 6 | Paso 4 · Duplicados | Ver antes de borrar |
| 7 | Paso 5 · Texto | De 122 departamentos a 33, y el problema que eso crea |
| 8 | Paso 6 · Valores imposibles | La parte que pandas no puede hacer por usted |
| 9 y 10 | Verificación, preguntas frecuentes y resumen | El workflow completo, listo para el reto |

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. Escribir código es el bloque 3, con el reto, y es
lo que se entrega.

**Las trece preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se
responden escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque
plegable *"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.**
Abrirlo antes no le ahorra nada: lo que se evalúa en el reto y en la sustentación es que usted sepa
mirar el resultado de una limpieza y decir qué decidió y por qué, no que sepa reconocer una
respuesta correcta cuando la ve.

**Las cajas "Para entender qué está pasando"** van en bloque citado, con la barra vertical a la
izquierda. Explican la herramienta por debajo de lo que se está haciendo: qué es una lista, qué hace
un `for`, por qué un método devuelve una tabla nueva en vez de cambiar la que tiene. **No son
materia de analítica, son el piso para entenderla.** Si ya programó antes, sáltelas sin culpa; si
nunca lo hizo, son la diferencia entre entender y copiar. Lo que ya se explicó en la clase 2
—pandas, DataFrame, Series, índice, método, máscara booleana— se referencia en una línea y no se
repite.

**Si algo se rompe**, muchas veces es a propósito: hay celdas que provocan el error con
`try / except` y lo explican al lado. En una clase de limpieza, los errores **son** el contenido.

**Punto de control:** al final hay tres preguntas para responderse a sí mismo antes de pasar al
reto.

---

## 0. Qué es limpiar datos, y por qué es la clase más importante del semestre

**Limpiar datos** es dejar un dataset en condiciones de responder preguntas sin mentir. No es
"ponerlo bonito": es que cada columna sea del tipo que dice ser, que cada valor sea posible, que
cada fila cuente una sola vez, y que lo que falta esté declarado como faltante en vez de disfrazado
de cero.

**La analogía de la lavandería**, que es la de toda la clase: nadie mete la ropa a la lavadora sin
mirar. Primero se vacía la bolsa: separar colores de blancos, sacar la moneda del bolsillo, revisar
si hay una media suelta. Eso es la **inspección**, y es el paso que todo el mundo se salta. Después
se lava, y cada problema tiene su tratamiento: la mancha no se trata igual que el botón suelto. Y
al doblar uno descubre que faltan medias. Siempre faltan medias.

**El dato duro del oficio:** un analista pasa entre el 60% y el 80% de su tiempo limpiando y el
resto analizando. No es un defecto de la profesión, es la profesión. Un modelo entrenado con datos
sucios produce conclusiones sucias, y nadie va a auditar el modelo: van a auditar la conclusión, y
va a estar su nombre en ella.

**Los 5 problemas** que se ven hoy, que son los mismos cinco de cualquier dataset del mundo:

| # | Problema | Imagen de la lavandería | Se ve como |
|---|----------|-------------------------|------------|
| 1 | **Valores nulos** | Faltan medias del par | Celdas vacías, `NaN` |
| 2 | **Tipos incorrectos** | Un zapato en la pila de camisas | Un número guardado como texto |
| 3 | **Duplicados** | Contar la misma camisa dos veces | La misma fila repetida |
| 4 | **Inconsistencias de texto** | Etiquetas que dicen "Azul", "AZUL", "  azul  " | Tres departamentos donde hay uno |
| 5 | **Valores inválidos de dominio** | Una camisa talla -3 y otra talla 250 | Un porcentaje de 118% o negativo |

Los problemas 3, 4 y 5 tienen algo en común y conviene decirlo ya: **no producen ningún error**. El
código corre perfecto y el resultado está mal. Son los caros.

---

## 1. El dataset, antes de tocarlo

Regla de la casa, de la clase 2: nunca se ejecuta una línea sobre un dataset que no se sabe qué es.

**Origen:** Ministerio de Educación Nacional, publicado en datos.gov.co. Estadísticas de educación
por departamento y año, entre 2011 y 2024.

**Forma:** 482 filas x 37 columnas. Una fila = un departamento en un año.

| Columna | Qué es |
|---------|--------|
| `ano` | Año del reporte |
| `departamento` | Departamento |
| `poblacion_5_16` | Población en edad escolar (5 a 16 años) |
| `cobertura_neta` | % de niños en edad escolar matriculados en el grado que les corresponde |
| `cobertura_bruta` | % de matriculados de **cualquier** edad sobre la población en edad escolar |
| `desercion` | % de estudiantes que abandonaron |
| `aprobacion`, `reprobacion`, `repitencia` | % de estudiantes en cada situación |

Las columnas se repiten con sufijo por nivel: `_transicion`, `_primaria`, `_secundaria`, `_media`.

**Qué tiene de sucio.** Los 5 problemas, todos a la vez:

| # | Problema | Dónde |
|---|----------|-------|
| 1 | Nulos | 12 columnas. Las dos peores rondan el 50% |
| 2 | Tipos | `ano` es decimal (`2023.0`). `poblacion_5_16` es texto porque trae comas y el literal `sin dato` |
| 3 | Duplicados | Hay filas exactamente repetidas |
| 4 | Texto | `departamento` tiene **122** valores únicos donde deberían ser 33 |
| 5 | Inválidos | Porcentajes negativos y porcentajes por encima de 100 |

Un aviso que va a importar al final: **`cobertura_bruta` puede legítimamente pasar de 100%**,
porque cuenta matriculados de cualquier edad sobre la población en edad escolar. `cobertura_neta`
no puede. Dos columnas que se ven igual, con reglas distintas.

**Este mismo dataset se vuelve a usar en la clase 13**, para estadística inferencial. La limpieza
de hoy no es un ejercicio desechable: es trabajo que se reutiliza.

### Setup

> **Para entender qué está pasando · lo que ya se explicó en la clase 2**
>
> Nada de esto se vuelve a explicar hoy. Si alguno se le borró, está desarrollado en el demo de la
> clase 2, en la sección que se indica.
>
> | Idea | En una línea | Dónde |
> |------|--------------|-------|
> | **Librería** | Código que alguien más escribió y publicó para no reinventarlo. Se instala una vez y se **importa** en cada archivo | Sección 1 |
> | **`import pandas as pd`** | "Trae pandas y llámalo `pd` aquí adentro". `pd` no es obligatorio para Python, es convención universal: toda la documentación del mundo lo usa | Sección 1 |
> | **pandas** | La librería estándar para trabajar datos en tablas con código. Nació en 2008 para hacer con código lo que se hacía a mano en Excel | Sección 1 |
> | **DataFrame** | Una tabla **en memoria**: filas, columnas con nombre, y un **índice** que etiqueta cada fila. Cada columna es una **Series** y tiene su propio tipo | Secciones 2, 3 y 5 |
> | **Método** | Una función que le pertenece a un objeto y se llama con punto y paréntesis: `df.head()` | Sección 3 |
> | **Máscara booleana** | Una Series de `True`/`False`, una decisión por fila, que se pasa entre corchetes para filtrar | Sección 7 |
>
> Dos consecuencias de "en memoria" que hoy importan más que nunca: **el CSV en disco nunca se
> toca** (todo lo que rompa se arregla volviendo a ejecutar `read_csv`), y **si cierra el cuaderno,
> el DataFrame se pierde**. La limpieza no se guarda sola: vive en las celdas, y por eso el cuaderno
> tiene que correr entero de arriba a abajo.

**Qué es `.copy()`.** `df_original = df.copy()` hace una copia independiente del DataFrame. No es
un respaldo por miedo: es el término de comparación. Al final vamos a poner el antes y el después
uno al lado del otro, y sin la copia no hay antes. Ojo con la alternativa: `df_original = df` **no**
copia nada; deja dos nombres apuntando a la misma tabla, y todo lo que le haga a una le pasa a la
otra.

Sobre la ruta: `../datos/` sube **dos** niveles, porque este dataset es compartido por varias
clases y vive en la raíz del repositorio. El del reto sube solo uno. No es un descuido, es la
convención de la casa, y es el error de ruta más común del semestre.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../datos/educacion_estadisticas.csv')
df_original = df.copy()

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

---

## 2. La deuda de la clase 2: dos formas de filtrar que faltaban

Antes de limpiar, cerramos lo que quedó anunciado.

### `.str.contains()` — filtrar por fragmento de texto

`==` pregunta "¿es exactamente esto?". `.str.contains()` pregunta "¿contiene esto?". Sirve para lo
que `==` no puede: encontrar todos los departamentos cuyo nombre contenga `SANTANDER` (hay dos:
Santander y Norte de Santander), o todos los registros de un año cuando la fecha está guardada como
texto.

**Recuerde qué es `.str`** (clase 2): el accesorio de texto de pandas. Una columna de texto no es un
string suelto, es una Series con cientos de strings adentro; `.str` aplica la operación de texto a
todos de un golpe, sin escribir un bucle.

**La pregunta incómoda: ¿qué debe responder `.str.contains()` en una fila que no tiene texto, sino
`NaN`?** ¿Contiene `SANTANDER` un dato que no existe? Ni sí ni no.

La respuesta depende de cómo esté guardada la columna, y por eso conviene verla con los dos casos al
lado. La celda de abajo lo hace: primero sobre la columna real del CSV, después sobre una Series
construida a mano del tipo genérico `object`, que es lo que le va a aparecer al armar datos usted
mismo o al leer archivos con versiones más viejas de pandas.

In [ ]:
# Caso 1: la columna del CSV. pandas 3 la lee como tipo 'str' y resuelve la duda por usted:
# lo que falta cuenta como que no contiene nada. No hay error.
mascara_csv = df['departamento'].str.contains('SANTANDER')
print("Tipo de la columna:", df['departamento'].dtype)
print("Nulos dentro de la máscara:", int(mascara_csv.isnull().sum()))
print("Filas encontradas:", int(mascara_csv.sum()))

# Caso 2: una columna del tipo genérico 'object' con un hueco. Aquí la máscara sale con hueco,
# y una máscara con huecos no sirve para filtrar.
ejemplo = pd.Series(['SANTANDER', None, 'ANTIOQUIA'], dtype=object)
mascara_object = ejemplo.str.contains('SANT')
print()
print("Máscara con hueco:", mascara_object.tolist())

try:
    ejemplo[mascara_object]
except Exception as error:
    print("Al filtrar con ella ->", type(error).__name__)
    print("  ", error)

**La costumbre profesional es escribir siempre `na=False`**, que significa "las filas sin valor
cuentan como que no". No es un parche contra un error que hoy no aparece: es **decir explícitamente
qué debe pasar con lo que falta**, en vez de dejarlo al criterio de la versión de pandas que tenga
instalada la máquina donde corra su código el año que viene.

Y hay una lección de fondo, que es de lo que trata la clase: **`na=False` no arregla los nulos, los
esquiva.** La solución de verdad es limpiarlos.

> **Para entender qué está pasando · los parámetros con nombre**
>
> `na=False` no es un símbolo mágico: es un **parámetro con nombre**. Una función o un método puede
> recibir cosas de dos formas: por posición (`replace(',', '')`, donde el primero es lo que se busca
> y el segundo por qué se reemplaza) o **por nombre** (`na=False`), donde usted dice explícitamente
> cuál está pasando.
>
> Los parámetros con nombre casi siempre tienen un **valor por defecto**: si usted no los escribe,
> la función usa el suyo. Escribirlos es tomarle la decisión a la librería en vez de heredarla.
>
> Esta clase entera es un desfile de parámetros con nombre, y ninguno es decorativo:
>
> | Parámetro | Qué decide |
> |-----------|-----------|
> | `na=False` | Qué responde una búsqueda de texto en una fila sin valor |
> | `subset=['col']` | Sobre qué columnas mira `dropna` para decidir si borra la fila |
> | `errors='coerce'` | Si una conversión imposible revienta o produce `NaN` |
> | `keep=False` | Si `duplicated` marca todas las copias o solo las repetidas |
> | `regex=False` | Si el texto a reemplazar se lee literal o como expresión regular |
> | `ascending=False` | Si `sort_values` ordena de menor a mayor o al revés |
>
> Cómo se averigua qué parámetros acepta algo, sin buscar en internet: escriba `df.dropna?` en una
> celda y ejecute. Jupyter le abre la documentación de la función, con sus parámetros y sus valores
> por defecto.

In [ ]:
santanderes = df[df['departamento'].str.contains('SANTANDER', na=False)]

print("Filas encontradas:", santanderes.shape[0])
print("Departamentos:", santanderes['departamento'].unique().tolist())

Fíjese en dos cosas del resultado: encontró **poquísimas** filas, y los nombres que sí encontró
están todos en mayúscula. Eso es el problema 4 asomando la cabeza: `.str.contains('SANTANDER')`
distingue mayúsculas, así que no ve `Santander` ni `santander`. Al final del cuaderno, después de
estandarizar el texto, el mismo filtro va a encontrar muchas más. Anótelo: **el número que le dio
este filtro está mal, y el código está bien.**

### `.query()` — la misma condición, escrita como una frase

`.query()` es una sintaxis alternativa para filtrar. En vez de repetir el nombre del DataFrame en
cada condición, se escribe la condición completa entre comillas, y ahí sí se usan las palabras
`and` y `or` en vez de `&` y `|`.

In [ ]:
# La forma que ya conoce
forma_conocida = df[(df['ano'] == 2023) & (df['desercion'] > 5)]

# La misma condición, con .query()
forma_query = df.query('ano == 2023 and desercion > 5')

print("Boolean indexing:", forma_conocida.shape[0], "filas")
print("Con .query():    ", forma_query.shape[0], "filas")

Mismo resultado, menos ruido para leer. **Por qué se aprende segundo y no primero:** `.query()`
construye la máscara por debajo, así que quien empieza por aquí no sabe depurar cuando algo falla; y
cuando falla, el mensaje de error es peor que el de boolean indexing. Además, los nombres de columna
con espacios o tildes le dan problemas (`Valor (Miles)` de la clase 2 no se puede escribir directo
en un `.query()`).

Úselo cuando la condición sea larga y simple. Para depurar, vuelva a la máscara.

---

## 3. Paso 1 · Inspeccionar

**Nunca limpie nada antes de saber qué tiene.** Es vaciar la bolsa y mirar antes de encender la
lavadora. El ritual son cinco comandos, siempre en el mismo orden, y produce un **diagnóstico
escrito**, que es lo que se entrega.

| Comando | Qué pregunta responde |
|---------|----------------------|
| `df.shape` | ¿De qué tamaño es esto? |
| `df.head()` | ¿Cómo se ven los datos? |
| `df.dtypes` | ¿Qué cree pandas que es cada columna? |
| `df.isnull().sum()` | ¿Qué falta, y dónde? |
| `df.describe()` | ¿Hay valores imposibles? |

In [ ]:
print("Forma:", df.shape)
print()
print("Columnas:")
print(df.columns.tolist())

In [ ]:
df.head()

`df.dtypes` devuelve el tipo de cada columna. Con 37 columnas, la salida completa no cabe en la
cabeza; `.head(10)` alcanza para el diagnóstico.

**Recordatorio de tipos** (clase 2): `int64` es entero, `float64` es decimal y admite `NaN`, `str`
es texto. Un número guardado como `str` no se puede sumar ni comparar de forma confiable.

In [ ]:
df.dtypes.head(10)

Dos cosas sospechosas ya se ven:

- **`ano` es `float64`.** Un año no tiene decimales; que salga `2023.0` es señal de que pandas
  encontró algo que le impidió leer la columna como entero. Ese "algo" casi siempre es un `NaN`.
- **`poblacion_5_16` es `str`** (texto), cuando debería ser un número. Alguna celda trae caracteres
  que pandas no supo convertir, y ante la duda pandas no adivina: guarda todo como texto.

### Los nulos

**Qué es `NaN`** (*Not a Number*): la marca que usa pandas para **"aquí no hay valor"**. Y aquí va
la distinción que hay que grabar, porque es el error conceptual más caro de la clase:

| No es lo mismo | Qué significa |
|----------------|---------------|
| `NaN` | **Desconocido.** Nadie midió, nadie reportó, se perdió el dato |
| `0` | **Un dato.** Alguien midió y el resultado fue cero |
| `''` (texto vacío) | **Una etiqueta en blanco.** Existe, pero no dice nada |

La analogía: meta la mano al cajón de las medias buscando una. No hay ninguna. Eso es `NaN`. No es
lo mismo que "tengo cero medias", que es el resultado de haber contado.

Consecuencia práctica, y dígala en voz alta: **si rellena una tasa de deserción faltante con 0,
acaba de afirmar que en ese departamento no desertó nadie.** Es una afirmación fuerte, probablemente
falsa, y se va a arrastrar hasta la conclusión final del informe.

**`df.isnull()`** devuelve un DataFrame del mismo tamaño lleno de `True` y `False`, y `.sum()`
cuenta los `True` por columna. Es exactamente el truco de la máscara booleana de la clase 2: `True`
vale 1 y `False` vale 0.

In [ ]:
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

print("Total de celdas vacías:", int(df.isnull().sum().sum()))
print()
print(nulos.to_string())

### Los valores imposibles

`describe()` da el resumen estadístico de las columnas numéricas. Para detectar suciedad, las dos
filas que importan son **`min` y `max`**: ahí es donde aparecen la edad de 200 años, el precio
negativo y la cobertura del 180%.

In [ ]:
df[['ano', 'cobertura_neta', 'cobertura_bruta', 'desercion', 'aprobacion']].describe().round(2)

**Pregunta de interpretación 1.** Mire la fila `min` y la fila `max`. ¿Hay algún valor que sea
imposible para un porcentaje? Escriba cuál y por qué es imposible.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Hay dos clases de imposibles a la vista. Los **negativos**: `desercion` y `reprobacion` bajan de
cero, y un porcentaje de estudiantes no puede ser negativo, porque no existe una cantidad negativa
de personas. Y los **mayores que 100**: `cobertura_neta` pasa de 100, y por definición no puede,
porque cuenta un subconjunto (los matriculados en el grado que les corresponde) sobre el total de
esa edad.

`cobertura_bruta` también pasa de 100, y ese **no** es un error: cuenta matriculados de cualquier
edad sobre la población en edad escolar, así que un departamento con muchos estudiantes en extraedad
la tiene por encima de 100 legítimamente. Dos columnas que se ven idénticas en `describe()`, con
reglas distintas. Eso no lo sabe pandas.

</details>

### Cuántas columnas están afectadas

Arriba salió cuántas **celdas** están vacías. Esa no es la cifra que se pone en un diagnóstico: para
decidir hace falta saber cuántas **columnas** están tocadas, porque el marco de decisión se aplica
columna por columna, no celda por celda.

`df.isnull().sum()` devuelve una Series con un conteo por columna. Sobre esa Series se vuelve a
preguntar con una máscara booleana, exactamente como en la clase 2: `> 0` la convierte en `True` y
`False`, y `.sum()` cuenta los `True`.

In [ ]:
nulos_por_columna = df.isnull().sum()
n_columnas_con_nulos = int((nulos_por_columna > 0).sum())

print("Columnas con al menos un nulo:", n_columnas_con_nulos, "de", df.shape[1])
print("Celdas vacías en total:       ", int(df.isnull().sum().sum()))
print()
print("Las cinco columnas más afectadas:")
print(nulos_por_columna.sort_values(ascending=False).head(5).to_string())

**Pregunta de interpretación 2.** Salieron dos cifras: cuántas columnas están afectadas y cuántas
celdas están vacías en total. ¿Por qué la decisión de limpieza se toma con la primera y no con la
segunda? Piense qué acción concreta le permite tomar cada número.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque el total de celdas vacías no se puede accionar. Es un número agregado sobre 37 columnas con
significados distintos: no dice si el hueco está en una columna que usted va a promediar o en una
que ni siquiera va a mirar, y no sugiere ninguna acción. Con "hay 1.500 celdas vacías" no se hace
nada. Con "12 columnas están afectadas, y estas dos rondan el 50%" ya hay una lista de decisiones
pendientes.

La razón de fondo es que **la unidad de decisión en limpieza es la columna**, no la celda. El marco
del paso 2 —eliminar, rellenar, o botar la columna— se aplica una vez por columna, y cada columna
tiene su propio dominio, su propio porcentaje y su propio significado del faltante. Una celda vacía
en `desercion` y una en `sedes_conectadas_a_internet` son dos problemas distintos que se resuelven
distinto, y sumarlos las vuelve indistinguibles.

Esto se repite todo el semestre: el número que sirve es el que se puede convertir en una acción. En
la clase 9, cuando se hable de narrativa, la regla vuelve con otro nombre: una cifra que no cambia
ninguna decisión no va en la diapositiva.

</details>

### El diagnóstico

Esto es lo que se entrega, no los comandos. Antes de escribir una sola línea de limpieza, el
diagnóstico queda por escrito:

| # | Problema | Evidencia | Comando que lo reveló |
|---|----------|-----------|------------------------|
| 1 | Nulos | 12 columnas con vacíos, dos rondando el 50% | `isnull().sum()` |
| 2 | Tipos | `ano` es float, `poblacion_5_16` es texto | `dtypes` |
| 3 | Duplicados | Por confirmar en el paso 4 | `duplicated()` |
| 4 | Texto | Por confirmar en el paso 5 | `nunique()` |
| 5 | Inválidos | Porcentajes fuera del rango 0-100 | `describe()` |

Fíjese en las dos filas que dicen "por confirmar": el diagnóstico inicial no lo ve todo, y no pasa
nada. Se escribe con lo que se sabe y se completa al pasar por cada paso.

---

## 4. Paso 2 · Valores nulos

Primero, cuantificar. **El número absoluto no sirve para decidir; el porcentaje sí.** 240 nulos es
un desastre en un dataset de 300 filas y una anécdota en uno de 100.000.

In [ ]:
pct_nulos = (df.isnull().sum() / len(df) * 100).round(1)
pct_nulos = pct_nulos[pct_nulos > 0].sort_values(ascending=False)

print(pct_nulos.to_string())

### El marco de decisión

Este es el concepto que se evalúa en el Momento 1. No es un comando: es un **criterio**.

| Nulos en la columna | Acción | Por qué |
|---------------------|--------|---------|
| Más del 50% | Considere eliminar la columna | Más huecos que datos: lo que rellene es invento, no dato |
| Menos del 5% | Elimine las filas afectadas | Pierde poquísimo y no inventa nada |
| Entre 5% y 50% | Rellene, y el con qué depende del dominio | Perder tantas filas es demasiado; hay que estimar |

**Tres advertencias sobre esta tabla:**

1. **Son umbrales orientativos, no leyes.** Una columna con 60% de nulos que es el corazón de su
   pregunta de investigación no se bota: se busca otra fuente o se cambia la pregunta. Una columna
   con 3% de nulos que a usted no le sirve para nada se puede botar entera sin drama.
2. **La zona del 5% al 50% es donde vive el criterio profesional.** Rellenar con la mediana no es
   "la respuesta correcta": es una decisión que usted toma y defiende.
3. **Toda decisión se escribe.** No en un comentario críptico: en una celda markdown, en español,
   diciendo qué hizo y por qué. Es lo que separa una limpieza defendible de una limpieza mágica.

Ahora aplicamos el marco columna por columna, **justificando cada decisión antes de escribirla**.

### Decisión 1 · `tamano_promedio_grupo` y `sedes_conectadas_a_internet` → rellenar con 0

Están cerca del 50%, en el borde: el marco dice "considere eliminarlas". No lo hacemos, y la razón
importa más que la decisión: **el patrón de los nulos no es aleatorio.** El dato dejó de reportarse
después de cierto año. No falta al azar, es que ese periodo no se midió.

Rellenamos con 0 con el significado "no disponible en este periodo", y lo dejamos documentado para
que nadie interprete ese 0 como una medición real. Es una decisión discutible y por eso va escrita.

In [ ]:
print("Antes:")
print("  tamano_promedio_grupo:      ", int(df['tamano_promedio_grupo'].isnull().sum()), "nulos")
print("  sedes_conectadas_a_internet:", int(df['sedes_conectadas_a_internet'].isnull().sum()), "nulos")

df['tamano_promedio_grupo'] = df['tamano_promedio_grupo'].fillna(0)
df['sedes_conectadas_a_internet'] = df['sedes_conectadas_a_internet'].fillna(0)

print()
print("Después:")
print("  tamano_promedio_grupo:      ", int(df['tamano_promedio_grupo'].isnull().sum()), "nulos")
print("  sedes_conectadas_a_internet:", int(df['sedes_conectadas_a_internet'].isnull().sum()), "nulos")

### Decisión 2 · `departamento` nulo → eliminar la fila

`departamento` es un identificador. Una fila sin departamento es una carta sin dirección: no se
puede usar para nada y no se puede adivinar. Está por debajo del 5%, así que el marco dice eliminar
las filas.

**`df.dropna(subset=['columna'])`** elimina solo las filas donde **esa** columna es nula. El
parámetro `subset` es el que hace la diferencia: `dropna()` a secas elimina toda fila que tenga
algún nulo en **cualquiera** de las 37 columnas. Véalo antes de usarlo, porque es un desastre
silencioso: no da error, solo devuelve un DataFrame diminuto.

> **Para entender qué está pasando · por qué se escribe `df = df.dropna(...)`**
>
> Fíjese en que `df` aparece **dos veces** en esa línea. No es un descuido, y entenderlo evita el
> error más frustrante de esta clase: "ejecuté la limpieza y el DataFrame quedó igual".
>
> **Qué es una variable:** un nombre pegado a un objeto que vive en memoria. `df` no *es* la tabla:
> es una etiqueta que apunta a ella.
>
> **Qué hace `=`:** despega la etiqueta de donde estaba y la pega a lo que haya al lado derecho.
> Primero se calcula todo lo de la derecha, y al final se reasigna el nombre.
>
> **Y la regla de pandas que lo explica todo:** casi ningún método modifica la tabla que tiene entre
> manos. `df.dropna(...)` **construye una tabla nueva** y se la devuelve; la original queda intacta.
> Si usted no guarda esa tabla nueva en ningún lado, Jupyter la imprime y se pierde. De ahí las dos
> formas que va a ver todo el semestre:
>
> ```python
> df = df.dropna(subset=['col'])       # reemplazo la tabla entera por la nueva
> df['col'] = df['col'].fillna(0)      # reemplazo una columna dentro de la tabla
> ```
>
> La consecuencia práctica: **una celda de limpieza sin `=` no limpia nada.** Y la contraria, que
> también muerde: una celda con `=` que se ejecuta dos veces limpia dos veces, y eso a veces cambia
> el resultado (borrar duplicados dos veces no hace daño; restarle algo a una columna dos veces sí).
> Por eso el cuaderno se corre de arriba a abajo, una vez.

In [ ]:
print("Filas actuales:               ", len(df))
print("Si usara dropna() sin subset: ", len(df.dropna()), "  <- desastre")

df = df.dropna(subset=['departamento'])

print("Con dropna(subset=[...]):     ", len(df))

### Decisión 3 · Columnas de tasas → rellenar con la mediana

Están entre 5% y 50%: hay que estimar. Rellenamos cada una con **su propia mediana**.

**Qué es la mediana:** el valor del medio cuando se ordenan todos los datos. **Por qué la mediana y
no el promedio:** el promedio se deja arrastrar por los valores extremos y la mediana no. Si en una
columna de tasas hay un 500 por error, el promedio sube para todas las filas que usted rellene, y
acaba de contaminar el dataset con un error que ya estaba. La mediana ni se entera. Es la analogía
del salario del CEO: si a una empresa de diez personas entra un CEO que gana cincuenta millones, el
promedio se dispara y la mediana no se mueve.

**Por qué no rellenamos con 0 aquí:** una deserción de 0% significa "no desertó nadie". La mediana
dice algo mucho más modesto y mucho más honesto: "si no lo sé, asumo que fue lo típico".

> **Para entender qué está pasando · la lista y el `for`**
>
> Son diez columnas con el mismo tratamiento. Escribir diez veces la misma línea cambiando el nombre
> es diez oportunidades de escribir mal una. La celda de abajo usa dos construcciones de Python que
> no habían aparecido:
>
> **Una lista** es una colección ordenada de valores, entre corchetes y separados por comas. Ya la
> usó sin nombrarla en la clase 2: los corchetes internos de `df[['col1', 'col2']]` y lo que recibe
> `.isin([...])` son listas. Aquí la lista es de nombres de columna, y se guarda en una variable
> para poder leerla y corregirla en un solo lugar.
>
> **Un `for`** recorre una colección y ejecuta el mismo bloque una vez por elemento:
>
> ```python
> for col in columnas_tasa:      # col vale 'cobertura_neta', luego 'desercion', ...
>     mediana = df[col].median()     # el bloque indentado se repite
>     df[col] = df[col].fillna(mediana)
> ```
>
> Tres detalles que rompen a todo el mundo la primera vez: `col` es un **nombre que usted inventa**
> (podría llamarse `columna` o `x`); los dos puntos al final de la línea del `for` son obligatorios;
> y **la indentación es la que define qué se repite**. Lo que está indentado se ejecuta una vez por
> columna; lo que vuelve al margen se ejecuta una sola vez, al final.
>
> Fíjese en el detalle que importa para el análisis: `df[col].median()` se calcula **adentro** del
> bucle, así que cada columna se rellena con **su propia** mediana. Sacarlo del bucle rellenaría
> todas las columnas con la mediana de una sola, y eso no sería un error de sintaxis: sería un
> dataset arruinado sin un solo mensaje en rojo.

In [ ]:
columnas_tasa = [
    'tasa_matriculacion_5_16',
    'cobertura_neta', 'cobertura_neta_primaria', 'cobertura_neta_secundaria',
    'desercion', 'desercion_transicion', 'desercion_primaria', 'desercion_secundaria',
    'aprobacion', 'reprobacion',
]

for col in columnas_tasa:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)

restantes = df.isnull().sum()
restantes = restantes[restantes > 0]

print("Columnas que todavía tienen nulos:")
print(restantes.to_string() if len(restantes) else "  ninguna")

Queda `poblacion_5_16`, y no se arregla aquí a propósito: su problema no es solo que falten valores,
es que **la columna entera está guardada como texto**. Rellenar texto con la mediana de un texto no
significa nada. Primero hay que convertirla, y eso es el paso 3.

**Pregunta de interpretación 3.** Supongamos que hubiéramos rellenado `desercion` con 0 en vez de la
mediana. Un informe posterior calcula la deserción promedio nacional. ¿En qué dirección se
equivocaría el informe, y por qué?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

El informe **subestimaría** la deserción: la reportaría más baja de lo que es. Cada 0 que metimos
entra al promedio como si fuera una medición real de "aquí no desertó nadie", y arrastra el promedio
hacia abajo. Con 47 filas rellenadas de 471, no es un detalle decorativo.

Lo grave no es el error de cálculo, es la dirección: el informe diría que el problema es más pequeño
de lo que es, justo en la variable que mide el problema. Y quien lea el informe no tiene forma de
saberlo, salvo que usted haya escrito qué rellenó y con qué. Por eso la justificación no es
burocracia: es la única defensa del lector.

</details>

### Verificar el paso 2

Después de tres decisiones, toca comprobar que quedó lo que esperábamos. Es el hábito que hay que
adquirir: **verificar después de cada paso**, no al final. Un paso de limpieza sin verificación es
una suposición.

Se miran tres cosas: cuántas filas sobreviven, cuántas celdas vacías quedan **en todo el DataFrame**
(dos sumas: `isnull().sum()` cuenta por columna, la segunda junta las columnas), y en qué columna
quedaron las que quedan.

In [ ]:
filas_restantes = len(df)
celdas_vacias = int(df.isnull().sum().sum())

print("Filas al cargar:         ", len(df_original))
print("Filas ahora:             ", filas_restantes)
print("Celdas vacías al cargar: ", int(df_original.isnull().sum().sum()))
print("Celdas vacías ahora:     ", celdas_vacias)
print()

pendientes = df.isnull().sum()
pendientes = pendientes[pendientes > 0]
print("Dónde quedaron:")
print(pendientes.to_string() if len(pendientes) else "  en ninguna parte")

**Pregunta de interpretación 4.** El conteo de celdas vacías bajó muchísimo, pero **ninguna de las
tres decisiones del paso 2 recuperó un solo dato**. ¿Qué se ganó entonces, y qué se perdió? Nombre
las dos cosas.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Lo que se ganó es **poder calcular**. Con `NaN` regados, media, mediana y conteos se vuelven
resbaladizos: pandas los descarta en silencio y cada columna acaba resumida sobre un número distinto
de filas, sin que nadie lo note. Después del paso 2 todas las columnas tratadas se calculan sobre la
misma base, y el resultado es comparable entre ellas.

Lo que se perdió es **información sobre la incertidumbre**. Antes, un `NaN` decía en voz alta "aquí
nadie midió". Ahora ese hueco es un 0 o una mediana indistinguible de un dato real: la celda ya no
confiesa que es una invención nuestra. El dataset se ve mejor de lo que es, y esa apariencia de
solidez es justamente el riesgo.

De ahí sale la regla que atraviesa la clase: la justificación escrita no es burocracia, es lo único
que queda del `NaN` después de rellenarlo. Y una alternativa práctica que se usa en proyectos serios,
y que sirve para el Momento 1: crear una columna hermana `desercion_imputada` con `True`/`False`, de
modo que el rastro viaje pegado al dato en vez de vivir solo en un párrafo.

</details>

---

## 5. Paso 3 · Tipos de datos

El zapato en la pila de camisas. Un número guardado como texto no permite hacer aritmética, no se
puede comparar con `>` de forma confiable, y desaparece de `describe()` sin avisar.

### `ano`: de decimal a entero

**Qué es `astype()`:** "trátame esta columna como si fuera de este otro tipo". Y trae una trampa
clásica que hay que ver una vez para reconocerla siempre.

In [ ]:
print("dtype:", df['ano'].dtype)
print("Muestra:", df['ano'].head(5).tolist())

**La trampa: `astype(int)` falla si hay `NaN`.** La razón es conceptual, no técnica: `NaN` es un
decimal, y no existe ningún número entero que signifique "desconocido". El entero 0 no sirve, porque
0 es un dato.

Y hay una segunda trampa, hermana: `astype(int)` sobre texto sucio también revienta, porque
`'sin dato'` no es ningún número.

Las dos celdas de abajo provocan los dos errores a propósito, atrapados con `try / except` para que
el cuaderno siga corriendo. **Lea el nombre del error**: son los dos que va a ver en el reto.

In [ ]:
# Error 1: convertir a entero con NaN presente
try:
    pd.Series([2011.0, 2012.0, np.nan]).astype(int)
except Exception as error:
    print("Con NaN ->", type(error).__name__)
    print("  ", error)

# Error 2: convertir a entero un texto que no es número
print()
try:
    pd.Series(['2011', '2012', 'sin dato']).astype(int)
except Exception as error:
    print("Con texto sucio ->", type(error).__name__)
    print("  ", error)

**La regla de orden, que resuelve las dos: rellenar primero, convertir después.** La secuencia segura
tiene tres partes:

1. `pd.to_numeric(serie, errors='coerce')` intenta convertir a número; lo que no pueda convertir, lo
   vuelve `NaN` **en vez de reventar**. Ese parámetro es el que hace la función utilizable con datos
   del mundo real: sin él, un solo `sin dato` tumba todo el proceso. Y convierte un error fatal en
   información, porque ahora usted puede **contar** cuántos problemas hay.
2. `fillna(...)` elimina los `NaN`, con la decisión que corresponda.
3. `astype(int)` ya puede convertir sin riesgo.

In [ ]:
df['ano'] = pd.to_numeric(df['ano'], errors='coerce').fillna(0).astype(int)

print("dtype:", df['ano'].dtype)
print("Muestra:", df['ano'].head(5).tolist())

### `poblacion_5_16`: de texto a entero

Primero mire **por qué** pandas la leyó como texto. Nunca convierta a ciegas: el valor que rompe la
conversión suele decirle algo del proceso que generó el archivo.

In [ ]:
print("dtype:", df['poblacion_5_16'].dtype)
print()
print("Muestra de valores:")
print(df['poblacion_5_16'].dropna().sample(12, random_state=42).tolist())

Ahí están los dos culpables: valores con **coma** como separador de miles (`401,539`) y el literal
`sin dato`. Cualquiera de los dos basta para que pandas guarde la columna entera como texto: ante la
duda, pandas no adivina ni descarta, guarda el valor tal cual.

### Convertir `poblacion_5_16`

Es la misma secuencia segura que acaba de ver para `ano`, con **un paso previo**: quitar las comas.

```python
serie.astype(str).str.replace(',', '', regex=False)   # quitar las comas
pd.to_numeric(serie, errors='coerce')                 # convertir; lo que no, NaN
serie.fillna(0).astype(int)                           # cerrar
```

`.astype(str)` de entrada es un seguro: garantiza que `.str` tenga texto con qué trabajar. Y
`regex=False` le dice a pandas que trate la coma como un carácter literal y no como una expresión
regular; con la coma da igual, con un punto no, y es mejor tener la costumbre.

La celda de abajo cuenta los nulos **antes** y **después** de `to_numeric`, y ese par de números es
lo que hay que mirar: el conteo sube. Los `sin dato` ya estaban ahí, disfrazados de texto; convertir
no creó el problema, lo hizo contable. Al final, `ano` y `poblacion_5_16` tienen que quedar las dos
en `int64`.

In [ ]:
print("Nulos antes de convertir:", int(df['poblacion_5_16'].isnull().sum()))

df['poblacion_5_16'] = pd.to_numeric(
    df['poblacion_5_16'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

print("Nulos después de to_numeric:", int(df['poblacion_5_16'].isnull().sum()),
      " <- aparecieron los 'sin dato'")

df['poblacion_5_16'] = df['poblacion_5_16'].fillna(0).astype(int)

print()
print("Final -> dtype de poblacion_5_16:", df['poblacion_5_16'].dtype)
print("        dtype de ano:            ", df['ano'].dtype)
print("Muestra:", df['poblacion_5_16'].head(5).tolist())

**Pregunta de interpretación 5.** El conteo de nulos **subió** al ejecutar `to_numeric`. Si en vez de
`errors='coerce'` hubiéramos escrito `astype(int)` directamente, ¿qué habría pasado, y por qué el
error sería preferible al silencio en un archivo pequeño pero no en uno de un millón de filas?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

`astype(int)` habría reventado con un `ValueError` en el primer `sin dato` que encontrara, y la
columna habría quedado sin tocar. Eso no es un fracaso: es la conversión negándose a inventar. El
problema es que el error se detiene en el **primer** valor problemático y no dice cuántos hay ni
dónde están, así que no permite decidir nada; solo permite ir a buscar a mano.

En un archivo de cincuenta filas eso está bien: uno lo abre, mira, y arregla el origen, que casi
siempre es la mejor solución. Con un millón de filas y una fuente que uno no controla, buscar a mano
no es una opción, y `errors='coerce'` convierte el error fatal en un dato: marca cada valor
problemático como `NaN` y el conteo de nulos pasa de 0 a "estos tantos". Ahí sí hay decisión posible:
si son cuatro, se botan las filas; si son el 40%, la columna no sirve y hay que hablar con quien
produce el archivo.

El precio de `coerce` es que **hay que mirar el conteo después**. Un `coerce` sin la comparación
antes/después es peor que el error, porque los valores problemáticos se vuelven `NaN`, después un
`fillna` los vuelve 0, y el archivo termina limpio y equivocado sin una sola línea en rojo. La regla
operativa: `errors='coerce'` y `print` del conteo van siempre juntos, en la misma celda.

</details>

**Pregunta de interpretación 6.** Rellenamos con 0 los `sin dato` de `poblacion_5_16`. Un 0 en
población escolar significa "en ese departamento no vive ningún niño", lo cual es falso. ¿Por qué se
puede aceptar aquí y no se aceptaba en `desercion`? ¿Qué haría usted en su lugar?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

En rigor **no es más aceptable**: es igual de falso. La diferencia es de uso, no de honestidad. A
`poblacion_5_16` la vamos a usar como identificador o como contexto, no la vamos a promediar para
sacar una conclusión; a `desercion` sí. Un 0 que nadie promedia hace menos daño que un 0 que entra a
una media nacional.

Dicho eso, la respuesta profesional es que ninguna de las dos debería quedar en 0. Las alternativas
razonables: dejar los faltantes como `NaN` y usar el tipo `Int64` de pandas (con I mayúscula), que
sí admite enteros con faltantes; o rellenar con el valor del mismo departamento en el año anterior,
que en una serie por año es una estimación mucho mejor que cero.

Que este cuaderno haya elegido el 0 y lo haya dejado escrito es exactamente el punto: **la decisión
mediocre declarada es auditable; la decisión buena escondida, no.**

</details>

---

## 6. Paso 4 · Duplicados

Contar la misma camisa dos veces. Un duplicado **no produce ningún error**: infla los conteos y
distorsiona los promedios en silencio. Si Antioquia aparece dos veces en 2023, Antioquia pesa el
doble en todo promedio nacional que usted calcule, y nadie lo va a notar mirando el resultado.

**`df.duplicated()`** devuelve una máscara booleana —la misma idea de la clase 2— que marca `True`
las filas que son copia de una anterior. Es decir: marca a partir de la **segunda** aparición.

In [ ]:
print("Filas duplicadas:", int(df.duplicated().sum()))

**Antes de borrar, mírelas.** `duplicated(keep=False)` marca **todas** las copias, incluida la
primera, que es lo que uno quiere para ver el problema completo: con `keep='first'` (el
comportamiento por defecto) usted ve una sola de las dos filas y no puede compararlas.

In [ ]:
copias = df[df.duplicated(keep=False)].sort_values(['departamento', 'ano'])

print("Filas involucradas en duplicados:", len(copias))
print()
print(copias[['ano', 'departamento', 'desercion', 'aprobacion']].head(12).to_string())

### Eliminar y verificar

**`drop_duplicates()`** se queda con la primera aparición y borra el resto. Se guarda el número de
filas antes, se elimina, y se resta: sin ese "antes" no hay forma de decir cuánto se borró, y "cuánto
se borró" es parte del diagnóstico.

Ojo con una cosa que va a morder a alguien: esta celda **no se puede ejecutar dos veces**. La segunda
vez ya no hay duplicados y `n_eliminadas` da 0. Si le pasa, vuelva arriba y ejecute todo de nuevo
desde `read_csv`. Esa es una de las razones por las que un cuaderno tiene que correr entero de arriba
a abajo.

In [ ]:
antes = len(df)
df = df.drop_duplicates()
n_eliminadas = antes - len(df)

print("Antes:     ", antes)
print("Después:   ", len(df))
print("Eliminadas:", n_eliminadas)
print()
print("Duplicados que quedan:", int(df.duplicated().sum()))

**Pregunta de interpretación 7.** Esta celda da un resultado distinto si se ejecuta dos veces, y en
cambio la del `fillna` de la mediana da lo mismo se ejecute las veces que se ejecute. ¿Qué distingue
a las celdas que se pueden repetir de las que no, y por qué eso obliga a correr el cuaderno completo
de arriba a abajo?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La diferencia está en si la celda depende del **estado previo** del DataFrame o solo de su contenido.
`fillna(mediana)` en la segunda pasada no encuentra ningún `NaN` que rellenar, así que no hace nada:
el resultado es el mismo. `n_eliminadas = antes - len(df)` sí depende del estado, porque `antes` se
mide sobre el `df` que ya viene modificado por la ejecución anterior. El código no cambió; cambió la
tabla sobre la que corre.

El detalle traicionero es que **el número equivocado no se ve equivocado**. Un 0 en "eliminadas" es
un número perfectamente plausible que uno copia al informe pensando que el dataset no tenía
duplicados, cuando lo que pasó fue que ya los había borrado. No hay error, no hay advertencia: hay
una cifra falsa con toda la apariencia de un hallazgo.

De ahí la disciplina de Jupyter que vale para el resto del semestre: un cuaderno solo es creíble si
corre entero, en orden, desde un kernel reiniciado. Ejecutar celdas sueltas mientras uno explora está
bien; entregar sin reiniciar y correr todo, no. En el Momento 1 esto es literal: si el notebook del
equipo no reproduce sus propios números al correrlo de cero, los números no existen.

</details>

**Pregunta de interpretación 8.** ¿Cuándo un duplicado **no** es un error? Dé un ejemplo del mundo
real donde dos filas idénticas sean dos hechos distintos y correctos.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Cuando la **unidad de análisis** admite repetición. Si cada fila es una transacción, dos compras del
mismo producto, por el mismo valor, en la misma tienda y el mismo día son dos hechos reales
distintos: borrar una es perder plata de verdad. Lo mismo con lecturas de un sensor cada minuto, o
con dos pasajeros distintos en el mismo vuelo con la misma tarifa.

La pregunta correcta antes de borrar no es "¿hay filas iguales?", es **"¿qué representa una fila?"**.
Aquí una fila es un departamento en un año, y un departamento no puede estar dos veces en el mismo
año: por eso aquí sí es error. En un dataset de transacciones, la misma línea de código habría
borrado información real.

Cuando hay una columna que debería ser única (un id, una llave), lo correcto no es
`drop_duplicates()` a secas sino `drop_duplicates(subset=['id'])`, que compara solo esa columna.

</details>

---

## 7. Paso 5 · Inconsistencias de texto

Las etiquetas que dicen "Azul", "AZUL" y "  azul  ". Es el problema más traicionero de los cinco,
porque no produce ningún error **y además parece que funciona**: el código corre, el resultado sale,
y está mal repartido.

**`nunique()`** cuenta cuántos valores distintos tiene una columna. Es el detector de este problema:
uno sabe cuántos debería haber, mira cuántos hay, y la diferencia es la suciedad.

Colombia tiene 32 departamentos más Bogotá: 33 valores posibles.

In [ ]:
print("Valores únicos en 'departamento':", df['departamento'].nunique())
print("Deberían ser:                     33")
print()
print("Una muestra de lo que hay:")
for valor in sorted(df['departamento'].unique().tolist())[:20]:
    print(f"  '{valor}'")

Se ve de todo: espacios al inicio y al final (por eso los imprimimos entre comillas: sin ellas los
espacios son invisibles), mayúsculas, minúsculas, capitalizado, con tilde y sin tilde, y hasta
`bogota, d,c,` con comas donde van puntos.

Lo arreglamos en **tres rondas**, contando después de cada una. La lección está en cuánto baja el
contador en cada ronda: las dos primeras son recetas mecánicas, la tercera hay que decidirla mirando.

### Ronda 1 · Espacios y mayúsculas

`.str.strip()` quita los espacios de los extremos (no los de adentro) y `.str.upper()` pasa todo a
mayúsculas. Se **encadenan**: cada una devuelve una Series, y sobre esa Series se puede volver a
llamar `.str`.

Un detalle que se pregunta siempre: **¿mayúsculas o minúsculas?** Da exactamente lo mismo mientras
sea consistente en todo el proyecto. Aquí se usa mayúscula porque es la convención de la mayoría de
los datasets oficiales colombianos, y así el texto limpio se parece a la fuente.

> **Para entender qué está pasando · encadenar métodos**
>
> `df['departamento'].str.strip().str.upper()` se lee **de izquierda a derecha, como una tubería**:
> tome la columna, quítele los espacios, y a lo que salga de ahí páselo a mayúsculas.
>
> Funciona porque cada método **devuelve un objeto nuevo** (la regla del cuadro de arriba), y sobre
> ese objeto se puede volver a llamar otro método. No hay ningún truco: son llamadas normales, una
> tras otra, sin variables intermedias.
>
> ```python
> paso1 = df['departamento'].str.strip()   # exactamente lo mismo,
> paso2 = paso1.str.upper()                # escrito en dos pasos
> ```
>
> Cuando la cadena es larga se parte en varias líneas envolviéndola en paréntesis, que es lo que
> hacen las celdas de abajo. Es solo estilo: el paréntesis le dice a Python "esto todavía no
> terminó".
>
> **Y la regla de depuración:** cuando una cadena de cinco métodos no hace lo que espera, párala en
> pasos y mire el resultado de cada uno. Encadenar es para leer, no para adivinar.

In [ ]:
antes = df['departamento'].nunique()

df['departamento'] = df['departamento'].str.strip().str.upper()

print("Antes:  ", antes)
print("Después:", df['departamento'].nunique())

Bajó muchísimo de un golpe, pero todavía no llega a 33. Faltan las tildes: `NARIÑO` y `NARINO`
siguen siendo dos departamentos distintos para pandas, porque son dos textos distintos.

### Lo que cuesta una tilde

Antes de aplicar la receta, veamos **qué precio tiene no aplicarla**. La pregunta es de las que se
responden en cualquier informe: ¿en qué departamentos se va más gente del colegio?

Se agrupa por departamento, se promedia la deserción, y se ordena de mayor a menor. Pero **al lado
del promedio vamos a pedir cuántas filas lo sostienen**, que es la costumbre que salva de casi todos
los rankings falsos.

In [ ]:
ranking = (df.groupby('departamento')['desercion']
             .agg(promedio='mean', filas='size')
             .sort_values('promedio', ascending=False))

print(ranking.head(6).round(2).to_string())

Mírelo con cuidado antes de seguir. Hay algo raro:

- `GUAINIA` encabeza el ranking con un promedio altísimo, y lo sostiene **una sola fila**.
- Más abajo aparece `GUAINÍA`, con trece filas y un promedio bastante menor.

Son el mismo departamento. Alguien tecleó un año sin la tilde, y ese único año se separó en una
categoría propia. Como es un solo dato y encima es un año malo, **su "promedio" es ese dato suelto**,
que aquí resulta ser el más alto de Colombia.

Y lo mismo con `CAQUETA` frente a `CAQUETÁ`, `CHOCO` frente a `CHOCÓ`, `BOYACA` frente a `BOYACÁ`.

**El daño no es el conteo, es la conclusión.** Un gráfico de barras con el top 10 pondría a `GUAINIA`
de primero, más alto que todos, y sería falso dos veces: ni ese es su promedio, ni ese departamento
es el peor. Nadie va a notarlo mirando el gráfico, porque el gráfico se ve perfecto. Es exactamente
el error que la clase 10 va a encontrar otra vez cuando este mismo dataset se lleve a Plotly.

> **La regla que se lleva:** cuando una categoría con **una sola fila** encabeza un ranking,
> sospeche de la ortografía antes de creerle al dato. La columna `filas` es lo que hace visible el
> engaño; sin ella, el ranking se ve impecable.

### Ronda 2 · Tildes

`.str.normalize('NFKD')` descompone cada letra acentuada en letra base más acento; al codificar a
ASCII ignorando lo que no cabe, los acentos desaparecen y queda la letra sola. Es la receta estándar
para quitar tildes: no hay que entenderla por dentro hoy, hay que saber que existe y que se copia
tal cual.

Advertencia honesta: esto convierte `Ñ` en `N`, y hay dominios donde eso importa (nombres propios,
llaves de cruce con otra tabla). Aquí no, porque `NARINO` sigue identificando al mismo departamento.

In [ ]:
antes = df['departamento'].nunique()

df['departamento'] = (df['departamento']
                      .str.normalize('NFKD')
                      .str.encode('ascii', 'ignore')
                      .str.decode('utf-8'))

print("Antes:  ", antes)
print("Después:", df['departamento'].nunique())

Y ahora el ranking de hace un momento, sobre el texto ya sin tildes. Es la misma pregunta y son los
mismos datos: lo único que cambió es que cada departamento está escrito de una sola forma.

In [ ]:
ranking = (df.groupby('departamento')['desercion']
             .agg(promedio='mean', filas='size')
             .sort_values('promedio', ascending=False))

print(ranking.head(6).round(2).to_string())

`GUAINIA` sigue de primero, pero ahora con catorce filas detrás y un promedio muy distinto, y el
resto del podio cambió de orden. **La respuesta correcta se parecía a la incorrecta**, y esa es la
razón por la que este problema es el más peligroso de los cinco: no avisa.

Casi. Mire qué queda: `BOGOTA, D,C,` frente a `BOGOTA, D.C.`, puntuación inconsistente que ninguna
receta general resuelve.

### Ronda 3, la que hay que decidir mirando

**No toda limpieza se automatiza.** Las dos primeras rondas fueron recetas: se aplican igual a
cualquier columna de texto de cualquier dataset. Esta no. Lo que sobra aquí son las comas y los
puntos de `BOGOTA, D.C.`, y eso solo se descubre imprimiendo la lista y mirándola.

Se quitan los dos caracteres, y de paso se colapsan los espacios dobles que quedan al borrar la coma.
`regex=False` es obligatorio con el punto: en una expresión regular, `.` significa "cualquier
carácter", y sin ese parámetro la línea borraría la columna entera dejándola en blanco. Sin error,
por supuesto.

In [ ]:
antes = df['departamento'].nunique()

df['departamento'] = (df['departamento']
                      .str.replace(',', '', regex=False)
                      .str.replace('.', '', regex=False)
                      .str.replace('  ', ' ', regex=False))

print("Antes:  ", antes)
print("Después:", df['departamento'].nunique(), " <- objetivo: 33")
print()
print(sorted(df['departamento'].unique().tolist()))

**Pregunta de interpretación 9.** Las rondas 1 y 2 (espacios, mayúsculas, tildes) se pueden copiar a
cualquier proyecto sin mirar los datos. La ronda 3 no. ¿Qué tienen las dos primeras que no tiene la
tercera, y qué le dice eso sobre hasta dónde puede llegar un script de limpieza reutilizable?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Las dos primeras corrigen diferencias que **nunca** son información. Dos textos que solo se
distinguen por un espacio al final, por mayúsculas o por una tilde son el mismo valor en cualquier
dominio imaginable: no hay ningún dataset donde `ANTIOQUIA` y `Antioquia ` signifiquen cosas
distintas. Por eso se pueden aplicar a ciegas.

La ronda 3 borra caracteres que en otro contexto **sí** significan algo. Un punto es basura en
`BOGOTA, D.C.` y es el separador decimal en `12.5`; una coma es basura aquí y es el separador de
miles en `401,539`, tres celdas más allá en este mismo archivo. Lo que decide no es el carácter, es
qué representa la columna, y eso lo sabe usted, no el script.

El límite práctico, que es la respuesta a "¿puedo automatizar todo esto?": un script de limpieza
reutilizable llega hasta donde llega la ronda 2, y hasta el **diagnóstico** de todo lo demás —contar
nulos, listar valores únicos, marcar rangos imposibles—. La corrección de la ronda 3 y las decisiones
del paso 2 no se heredan de un proyecto a otro. Por eso el entregable del Momento 1 no es el código
de limpieza: son las decisiones justificadas.

</details>

### La lección que no se puede saltar

Volvamos a contar duplicados. Van a aparecer **nuevos**, y no es un fallo suyo.

Por qué: `Antioquia` y `ANTIOQUIA` en el mismo año eran, para pandas, dos filas distintas; en la
realidad eran una sola. Al estandarizar el texto, la realidad se hizo visible.

**Limpiar un problema crea otro.** La limpieza no es una lista de tareas que se tacha una vez: es
**iterativa**, y hay que volver a verificar después de cada paso. Si se lleva una sola idea de este
cuaderno, que sea esta. La regla operativa que se deriva: **los duplicados se revisan siempre
después de tocar el texto**, nunca antes y ya.

In [ ]:
print("Duplicados nuevos, después de estandarizar el texto:", int(df.duplicated().sum()))

antes = len(df)
df = df.drop_duplicates()

print("Filas antes:", antes)
print("Filas ahora:", len(df))

Y ahora, la prueba de que esto sirvió para algo: el filtro del principio del cuaderno, el de
`.str.contains('SANTANDER', na=False)`, sobre los datos limpios.

In [ ]:
ahora = int(df['departamento'].str.contains('SANTANDER', na=False).sum())

print("Filas con SANTANDER, antes de limpiar el texto:", santanderes.shape[0])
print("Filas con SANTANDER, después:                 ", ahora)

---

## 8. Paso 6 · Valores inválidos de dominio

La camisa talla -3 y la camisa talla 250. Valores que existen, que pandas acepta sin chistar, y que
son imposibles en el mundo real.

**Es el único de los cinco problemas que no se puede detectar sin conocer el dominio.** pandas no
tiene idea de que un porcentaje va de 0 a 100, de que una edad no es negativa, ni de que una tasa
por mil sí puede pasar de 100. Esta es la parte del trabajo que no se automatiza, y la razón por la
que le pagan a usted y no a un script.

### El caso de cobertura neta contra cobertura bruta

- **Cobertura neta:** matriculados que están en el grado que les corresponde por edad, sobre la
  población de esa edad. Es un subconjunto sobre el total: **no puede pasar de 100%**.
- **Cobertura bruta:** matriculados de **cualquier** edad sobre la población en edad escolar. Si un
  departamento tiene muchos estudiantes en extraedad, **sí puede pasar de 100%** legítimamente.

Dos columnas que se ven idénticas en un `describe()`, con reglas distintas. Si valida las dos con la
misma regla, "corrige" datos que estaban perfectos, y el error queda invisible para siempre.

**Cómo se averigua el rango válido de una columna:** no lo sabe pandas, lo sabe el dominio. Busque
el diccionario de datos de la fuente. Si la fuente no lo publica, eso también es información
valiosa sobre la fuente.

In [ ]:
columnas_porcentaje = [
    'cobertura_neta', 'cobertura_neta_primaria', 'cobertura_neta_secundaria',
    'desercion', 'desercion_primaria', 'desercion_secundaria',
    'aprobacion', 'reprobacion',
]

print(df[columnas_porcentaje].describe().loc[['min', 'max']].round(2).to_string())

In [ ]:
negativos = (df[columnas_porcentaje] < 0).sum()
sobre_100 = (df[columnas_porcentaje] > 100).sum()

print("Valores negativos:")
print(negativos[negativos > 0].to_string() if negativos.sum() else "  ninguno")
print()
print("Valores por encima de 100:")
print(sobre_100[sobre_100 > 0].to_string() if sobre_100.sum() else "  ninguno")
print()
print("Total de valores inválidos:", int(negativos.sum() + sobre_100.sum()))

### La decisión

Dos caminos defendibles:

- **(a) Convertirlos en `NaN` y rellenar con la mediana.** Asume que el valor real era desconocido y
  que lo típico es la mejor estimación disponible.
- **(b) Recortarlos al rango válido:** los negativos a 0, los mayores de 100 a 100. Asume que el
  valor real estaba en el extremo y que el error fue de medición o de digitación.

Elegimos **(a)**, y el argumento es este: un -5 en deserción no sugiere que la deserción real fuera
0; sugiere que alguien se equivocó y que no sabemos qué había. Tratamos el dato imposible igual que
un dato ausente, que es lo que es.

**La decisión importante no es cuál de los dos: es haberla escrito.** Un revisor puede estar en
desacuerdo con (a); no puede estar en desacuerdo con algo que no encuentra.

**Qué es `df.loc[condicion, 'columna'] = valor`:** es la forma de modificar el **original**, y viene
de la clase 2. Se lee como una frase: "en las filas que cumplen esta condición, en esta columna, pon
este valor". La alternativa que **no** funciona es asignar sobre el resultado de un filtro
(`df[condicion]['columna'] = valor`): eso modifica una tabla nueva que se descarta enseguida, el
original queda igual, y **no aparece ninguna advertencia**. El síntoma es el silencio.

In [ ]:
arreglados = 0

for col in columnas_porcentaje:
    invalidos = (df[col] < 0) | (df[col] > 100)
    n = int(invalidos.sum())
    if n > 0:
        df.loc[invalidos, col] = np.nan
        df[col] = df[col].fillna(df[col].median())
        arreglados += n
        print(f"{col:<28} {n} valores reemplazados por la mediana")

print()
print("Total arreglado:", arreglados)

In [ ]:
print(df[columnas_porcentaje].describe().loc[['min', 'max']].round(2).to_string())
print()
restantes = int((df[columnas_porcentaje] < 0).sum().sum() + (df[columnas_porcentaje] > 100).sum().sum())
print("Valores inválidos restantes:", restantes)

### La columna que NO se corrige

Fíjese en qué columna **no** apareció en la lista de arriba: `cobertura_bruta`. No es un olvido. Se
quedó fuera de `columnas_porcentaje` a propósito, porque su rango válido no es 0-100.

La celda de abajo la cuenta, con el patrón de conteo de la clase 2 —una condición sobre una columna
y `.sum()` sobre la máscara—, y **no la arregla**.

In [ ]:
n_bruta_sobre_100 = int((df['cobertura_bruta'] > 100).sum())

print("Valores de cobertura_bruta por encima de 100:", n_bruta_sobre_100, "de", len(df))
print("Porcentaje del dataset:", round(n_bruta_sobre_100 / len(df) * 100, 1), "%")
print("Máximo de la columna:", round(float(df['cobertura_bruta'].max()), 2))
print()
print("Comparación con cobertura_neta, ya corregida:")
print("  máximo de cobertura_neta: ", round(float(df['cobertura_neta'].max()), 2))

**Pregunta de interpretación 10.** Mire otra vez el bucle que corrigió los porcentajes inválidos:
primero pone los valores imposibles en `NaN` y **después** calcula `df[col].median()` para
rellenarlos. Si esas dos líneas se invirtieran —calcular la mediana primero y luego anular los
imposibles— el código correría igual, sin error. ¿Qué cambiaría en el resultado, y por qué ese tipo
de fallo es el más difícil de detectar?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La mediana quedaría **contaminada por los valores que estamos declarando imposibles**. Al calcularla
antes, entran al ordenamiento el -5 de deserción y el 180 de cobertura neta, y el valor del medio se
corre. Después se usa ese número corrido para rellenar, así que el error que se pretendía sacar del
dataset vuelve a entrar por la puerta de atrás, diluido en cada celda que rellenamos. Es la misma
lógica por la que se rellena con la mediana y no con el promedio, aplicada un nivel más arriba: no
basta con elegir un estadístico robusto, hay que calcularlo sobre datos que ya estén limpios.

En este dataset el desplazamiento sería pequeño, porque los valores imposibles son pocos frente al
total y la mediana es justamente el estadístico que menos se mueve. Ese es el punto: **es pequeño, no
es cero, y no se ve**. El cuaderno corre, la columna queda dentro de 0-100, `describe()` se ve
perfecto, y el sesgo viaja hasta la conclusión sin dejar rastro.

Por eso los errores de **orden** son la categoría más cara de esta clase, y por eso el workflow es
una secuencia y no una lista de tareas sueltas. Ya apareció dos veces hoy con la misma forma:
rellenar antes de convertir (`astype(int)` no soporta `NaN`) y revisar duplicados después de tocar el
texto. La regla que las une: **cada paso se calcula sobre el estado que dejó el anterior**, así que
cambiar el orden cambia el resultado aunque no cambie ni una línea de código.

</details>

**Pregunta de interpretación 11.** Son muchos. ¿Los dejaría así o los corregiría? Justifique en dos
frases, usando la definición de cobertura bruta.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Se dejan como están. La cobertura bruta cuenta matriculados de cualquier edad sobre la población en
edad escolar, así que en un departamento con muchos estudiantes en extraedad —repitentes, personas
que volvieron a estudiar de adultas— el numerador es legítimamente mayor que el denominador y el
resultado pasa de 100 sin que nadie se haya equivocado.

Y hay una señal adicional que confirma el diagnóstico: son cientos de valores, no un puñado. Un
error de digitación no afecta a la mitad del dataset de forma sistemática. Cuando un "valor
imposible" aparece masivamente, la primera hipótesis no es que los datos estén mal: es que uno
entendió mal qué mide la columna.

Corregirlos habría sido peor que no hacer nada: habría destruido la información real de extraedad, y
sin dejar rastro.

</details>

---

## 9. Verificación final: antes y después

La limpieza no está terminada hasta que se puede **demostrar**. Para eso guardamos `df_original` al
principio.

> **Para entender qué está pasando · definir una función**
>
> Queremos el mismo resumen dos veces, sobre dos tablas distintas. En vez de copiar seis `print`,
> se define **una función**: un bloque de código con nombre, que se escribe una vez y se ejecuta las
> veces que haga falta.
>
> ```python
> def resumen(datos, etiqueta):   # def, nombre, parametros entre parentesis, dos puntos
>     print(etiqueta)             # el cuerpo va indentado
>     print("  Filas:", len(datos))
>
> resumen(df_original, "ANTES")   # llamarla: el valor entra por el parametro
> resumen(df, "DESPUÉS")
> ```
>
> `datos` y `etiqueta` son **parámetros**: nombres provisionales que valen lo que usted le pase al
> llamarla. Dentro de la función, `datos` es `df_original` la primera vez y `df` la segunda. Que se
> llamen así y no `df` es a propósito: dentro de la función no importa cómo se llame la tabla
> afuera, y esa independencia es justo lo que la hace reutilizable.
>
> Definir la función **no ejecuta nada**: solo la deja guardada. Se ejecuta cuando se la llama.
>
> Esto es el primer paso hacia lo que dice la sección de preguntas frecuentes: la parte mecánica de
> la limpieza se automatiza en una función de diagnóstico. El criterio no.

In [ ]:
def resumen(datos, etiqueta):
    print(etiqueta)
    print("  Filas:                ", len(datos))
    print("  Celdas vacías:        ", int(datos.isnull().sum().sum()))
    print("  Filas duplicadas:     ", int(datos.duplicated().sum()))
    print("  Departamentos únicos: ", datos['departamento'].nunique())
    print("  dtype de ano:         ", datos['ano'].dtype)
    print("  dtype de poblacion:   ", datos['poblacion_5_16'].dtype)
    print()

resumen(df_original, "ANTES (original)")
resumen(df, "DESPUÉS (limpio)")

### Una pregunta de verdad sobre datos limpios

Hasta aquí limpiamos por limpiar. Pero la limpieza no es el entregable de nadie: es la condición
para que la respuesta valga algo. La pregunta: **¿en qué cinco departamentos se fue más gente del
colegio en 2023?**

Se responde con lo de la clase 2 más un comando nuevo:

```python
df[df['columna'] == valor]                  # filtrar
df[['col1', 'col2']]                        # elegir columnas
df.sort_values('columna', ascending=False)  # ordenar de mayor a menor
df.head(5)                                  # los cinco primeros
df['columna'].tolist()                      # pasar una columna a lista de Python
```

Ojo con `ascending`: por defecto ordena de menor a mayor, que es justo lo contrario de lo que pide la
pregunta. Un ranking al revés no da error y se ve igual de convincente.

In [ ]:
top5 = (df[df['ano'] == 2023][['departamento', 'desercion']]
        .sort_values('desercion', ascending=False)
        .head(5))

top5_desercion_2023 = top5['departamento'].tolist()

print("Deserción más alta en 2023:")
print(top5.to_string(index=False))
print()
print("Como lista:", top5_desercion_2023)
print()
print("Para comparar, la mediana nacional de desercion:",
      round(float(df['desercion'].median()), 2))

**Pregunta de interpretación 12.** Este mismo ranking, con este mismo código, se pudo haber calculado
antes de limpiar nada: el cuaderno habría corrido igual y habría impreso cinco departamentos. ¿En
cuáles de los seis pasos de hoy habría cambiado la respuesta, y qué le dice eso sobre por qué la
limpieza va antes del análisis y no después?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Habría cambiado en casi todos, y ya vimos uno en vivo: con el texto sucio, `GUAINIA` y `GUAINÍA` eran
dos departamentos, y el ranking ponía de primero a una categoría de una sola fila. Los valores
imposibles habrían metido un -5 o un 180 en la columna que se ordena, y como el orden es descendente,
un valor absurdo alto va derecho al primer puesto. Los duplicados habrían dejado un departamento
compitiendo dos veces por el top 5. Y los nulos habrían sacado del ranking a departamentos que no
tenían dato, sin decirlo.

Lo importante no es que la respuesta fuera distinta, es que **habría sido igual de presentable**.
Cinco nombres, cinco números con dos decimales, listos para la diapositiva. No hay ninguna señal en
la salida que permita distinguir el ranking bueno del malo: los dos se ven exactamente igual, y ese
es el argumento entero de la clase. Un análisis sobre datos sucios no falla, miente con buena
presentación.

Por eso el orden no es negociable y por eso el diagnóstico va por escrito. Y una consecuencia
práctica para el Momento 1: cuando el resultado de un análisis parezca raro, la primera hipótesis no
es que el hallazgo sea sorprendente, es que la limpieza no está terminada. Una categoría con una sola
fila encabezando un ranking es el caso de manual, y ya sabe cómo se detecta: pidiendo el conteo de
filas al lado del promedio.

</details>

**Pregunta de interpretación 13.** Escriba dos frases sobre lo que muestra ese resultado. Y una
tercera sobre **qué tanto confía en él**, dado todo lo que tuvo que rellenar para llegar hasta aquí.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La descripción es fácil: los cinco departamentos con mayor deserción en 2023 son en su mayoría
territorios de la periferia —amazónicos y de la Orinoquía— más algún departamento andino, y las
diferencias entre el primero y el quinto no son enormes. Eso es lo que dice la tabla.

La tercera frase es la que separa un análisis de una tabla bonita. La confianza tiene que ser
moderada, y por una razón concreta: `desercion` tenía cerca de un 10% de nulos que **nosotros**
rellenamos con la mediana. Si a un departamento le faltaba el dato de 2023 y le pusimos el valor
típico nacional, ese departamento aparece en el puesto que le dimos, no en el que le corresponde.

Lo honesto es mirar cuáles de esos cinco tenían el dato original y cuáles no, y decirlo. Un analista
que reporta un ranking sin mencionar que un décimo de la columna es estimación propia no está
mintiendo, pero está a un paso.

</details>

---

## 10. Punto de control

Este cuaderno no se autocalifica: no hay nada que teclear en él. El punto de control es usted
respondiéndose, sin abrir los desplegables, estas tres preguntas:

1. ¿Sabría hacer el ritual de inspección sobre un archivo que nunca vio y escribir el diagnóstico,
   con la evidencia al lado de cada problema?
2. ¿Sabría decidir qué hacer con una columna que tiene nulos —eliminar, rellenar, botarla— y
   **defender** esa decisión ante alguien que preferiría la contraria?
3. ¿Sabría explicar, sin mirar, por qué se rellena antes de convertir y por qué los duplicados se
   cuentan otra vez después de tocar el texto?

Si alguna respuesta es "no", no pase de largo: el reto empieza dando por sabido todo esto, y la clase
4 da por sabido el reto. Si está en el salón, levante la mano ahora, que el profesor está aquí para
eso.

---

## 11. Preguntas que siempre salen

**¿No es hacer trampa rellenar datos que no tengo?**
Es una decisión, y como toda decisión hay que declararla. Rellenar con la mediana es afirmar "si no
sé el valor, asumo que es el típico": es una suposición explícita y auditable. Lo que sí es trampa es
rellenar sin decirlo, o rellenar con un valor que cambia la conclusión y no mencionarlo.

**¿Cuándo elimino una columna en vez de rellenarla?**
Cuando tiene tantos huecos que lo que quedaría sería mayoritariamente invento suyo. El umbral
orientativo es 50%. Pero si esa columna es central para su pregunta de investigación, no la bote:
consiga otra fuente o cambie la pregunta. Un dataset donde la variable que le importa está vacía en
el 60% de los casos no es el dataset correcto para esa pregunta.

**¿Por qué `errors='coerce'` y no dejar que reviente?**
Porque reventar solo le sirve si el archivo es pequeño y usted puede revisarlo a mano. Con datos
reales, `coerce` convierte lo problemático en `NaN`, y entonces usted puede **contar** cuántos
problemas hay y decidir. Convierte un error fatal en información.

**¿Este workflow de 5 pasos hay que seguirlo siempre en ese orden?**
El paso 1 (inspeccionar) siempre va primero, sin excepción. El resto tiene un orden recomendado, pero
lo que importa es **verificar después de cada paso**. La regla dura no es el orden: es que los
duplicados se revisan al final, después de tocar el texto.

**¿Puedo automatizar todo esto en una función y olvidarme?**
Puede automatizar la parte mecánica: quitar espacios, normalizar mayúsculas, detectar duplicados
exactos, listar el porcentaje de nulos. No puede automatizar el criterio: qué columna vale la pena
conservar, con qué rellenar, qué rango es válido. Escribir esa función de diagnóstico es, de hecho,
una buena idea para el proyecto.

**¿Qué pasa si borro filas y el índice queda con huecos?**
Nada grave: el índice es una **etiqueta**, no una posición, como se anunció en la clase 2. Si le
molesta ver `0, 1, 4, 7`, existe `reset_index(drop=True)`. Lo importante es entender que después de
borrar filas, el índice y la posición dejaron de coincidir, y que `.iloc` (posición) y `.loc`
(etiqueta) ya no dan lo mismo.

**¿Los datos limpios de hoy me sirven para el proyecto?**
El dataset no; el workflow sí. Los CSV de las clases son material de enseñanza, elegidos por lo que
permiten enseñar. El entregable del Momento 1 empieza exactamente con estos 5 pasos aplicados al
dataset de su equipo, con las decisiones justificadas por escrito.

**¿Por qué tanto español en un curso de programación?**
Porque lo que se evalúa no es que el código corra: es que usted pueda decir qué decidió y por qué.
En limpieza de datos, más que en ningún otro tema, el código lo escribe cualquiera y la decisión la
defiende usted.

---

## 12. Resumen — el workflow de 5 pasos

Esto es lo que se lleva. Se aplica igual a cualquier dataset, incluido el de su proyecto.

| Paso | Qué | Comandos clave |
|------|-----|----------------|
| 1. **Inspeccionar** | Entender antes de tocar. Producir un diagnóstico escrito | `shape`, `head()`, `dtypes`, `isnull().sum()`, `describe()` |
| 2. **Nulos** | Cuantificar en %, aplicar el marco de decisión, justificar | `isnull().sum()`, `dropna(subset=)`, `fillna()` |
| 3. **Tipos** | Convertir a lo que debería ser. Rellenar antes de convertir | `pd.to_numeric(errors='coerce')`, `astype()`, `.str.replace()` |
| 4. **Duplicados** | Ver antes de borrar | `duplicated(keep=False)`, `drop_duplicates()` |
| 5. **Texto e inválidos** | Estandarizar, y **volver a revisar duplicados** | `.str.strip()`, `.str.upper()`, `.str.normalize()`, máscaras de rango |

**Las cuatro reglas que no se negocian:**

1. Inspeccionar siempre va primero. Sin diagnóstico escrito no se limpia nada.
2. Rellenar primero, convertir después. `astype(int)` no soporta `NaN`.
3. Después de tocar el texto, se vuelven a contar los duplicados. Siempre.
4. Toda decisión se justifica en español, en una celda markdown. La justificación vale más que el
   código.

**Autoevaluación honesta.** Si puede responder que sí a estas cinco, está listo para el reto:

- [ ] Puedo hacer el ritual de inspección y escribir un diagnóstico sin que nadie me lo pida.
- [ ] Sé explicar por qué `NaN` no es 0 y por qué eso cambia una conclusión.
- [ ] Sé aplicar el marco de decisión de nulos y **defender** la decisión que tomé.
- [ ] Sé por qué `astype(int)` revienta y en qué orden se arregla.
- [ ] Sé por qué hay que volver a contar duplicados después de limpiar el texto.

**Ahora: el reto.** `../reto/reto_starter.ipynb`, con indicadores de salud materno-infantil por
municipio. Los mismos 5 problemas, otro dataset, y una trampa de dominio esperándolo.